# 11 — Paired author CSC vs CSC + LoRA evaluation

Use an **inference** CUDA session. Restore the baseline, frozen candidate cache, split metadata, pair files, and trained adapter from 09–10. The generation model is not called again. Train database files are not needed for this stage; the prepared metadata/pair manifests remain required.

Set `CSC_CONFIG` to the saved config and `RUN_CSC_GPU=1` for pending LoRA merge work. `CSC_K` can select a prefix of the cached candidates (for example 16 from 64) without new generation. A prefix requires its own author CSC baseline before LoRA comparison; no K=16 latency is inferred from a K=64 generation run.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

configured = os.environ.get('SLM_PRJ_PATH')
candidates = ([Path(configured)] if configured else []) + [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/SLM-PRJ')]
REPO_DIR = next((p.resolve() for p in candidates if (p / 'research/slm_sql').is_dir()), None)
if REPO_DIR is None:
    REPO_DIR = Path('/kaggle/working/SLM-PRJ')
    subprocess.run(['git', 'clone', os.environ.get('SLM_PRJ_REPO_URL', 'https://github.com/minhadcaz/SLMSQL_project.git'), str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
if os.environ.get('INSTALL_CSC_DEPS', '0') == '1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(REPO_DIR / 'requirements-csc-inference.txt')], check=True)


In [ ]:
from scripts.csc_notebook import configure, run
CONFIG, cfg = configure(REPO_DIR)
SEED = int(os.environ.get('CSC_SEED', '42'))
RUN_GPU = os.environ.get('RUN_CSC_GPU', '0') == '1'
print({'config': str(CONFIG), 'experiment': cfg['experiment_dir'], 'seed': SEED, 'run_gpu': RUN_GPU})
run(REPO_DIR, CONFIG, 'plan', '--seed', str(SEED))


In [ ]:
# Restore only after the full experiment directory/config/pair manifests are in place.
if os.environ.get('RESTORE_CSC_ADAPTER', '0') == '1':
    from scripts.csc_notebook import restore_csc_adapter
    print(restore_csc_adapter(cfg))
else:
    print('Adapter restore is off; expecting experiment_dir/adapter/final.')


In [ ]:
K = int(os.environ.get('CSC_K', str(cfg['generation']['evaluation_samples'])))
if RUN_GPU:
    # For the original K this is a cache hit; for another prefix it runs only original CSC.
    run(REPO_DIR, CONFIG, 'merge', '--variant', 'author', '--seed', str(SEED), '--k', str(K))
    run(REPO_DIR, CONFIG, 'merge', '--variant', 'lora', '--seed', str(SEED), '--k', str(K))
else:
    print('Pending merge stages not started. No generator is invoked by this notebook.')


## Score cached predictions — CPU only

Reports SC, original CSC and LoRA CSC on exactly paired questions, BIRD set EX, difficulty/JOIN/aggregate slices, correction and regression counts, output-token costs, bootstrap CIs and paired McNemar tests. All-invalid merge output is a failure, not an unreported fallback to SC. Missing model outputs never become a fabricated zero accuracy.


In [ ]:
author_path = Path(cfg['experiment_dir']) / f'seed{SEED}' / f'evaluation.author.csc-k{K}.jsonl.manifest.json'
lora_path = Path(cfg['experiment_dir']) / f'seed{SEED}' / f'evaluation.lora.csc-k{K}.jsonl.manifest.json'
if all(path.exists() and json.loads(path.read_text()).get('complete') for path in (author_path, lora_path)):
    run(REPO_DIR, CONFIG, 'score', '--seed', str(SEED), '--k', str(K))
else:
    print('Complete paired CSC predictions are required before scoring.')


In [ ]:
report_path = Path(cfg['experiment_dir']) / 'reports' / f'evaluation-seed{SEED}-k{K}.comparison.json'
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps({'summary': report['summary'], 'comparison': report['comparison'], 'generation_diagnostics': report['generation_diagnostics']}, indent=2))
    improved = [r for r in report['records'] if r['correct']['lora_csc'] and not r['correct']['author_csc']]
    regressed = [r for r in report['records'] if r['correct']['author_csc'] and not r['correct']['lora_csc']]
    print({'improved': len(improved), 'regressed': len(regressed)})
else:
    print('No paired report exists yet.')
